# Stage 6 - Multivariable calculus visualization

Four plots that make the MV-calculus foundation of the trained classifier visible:

1. **Loss surface slice** in two first-layer weights (MV 2.6 level sets).
2. **Gradient norm vs. epoch** from `runs/training_log.csv` (critical-point approach).
3. **PCA of the 279-dim input space** (`R^279 -> R^2`).
4. **Chain rule verification** of `dL/dW^(1)_{ij}` against PyTorch autograd.

Helper functions live in `src/mv_visualization.py` so each cell stays narrow.

*Authoritative spec: `tasks/gesture_recognition_plan_v2.md` 6.4, `tasks/implementation_stages.md` Stage 6.*

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import evaluate as ev
from src import mv_visualization as mv
from src.train import DEFAULT_SEED, pick_device, set_global_seeds

set_global_seeds(DEFAULT_SEED)
DEVICE = pick_device('cpu')  # evaluation + visualisation run cleanly on CPU
EVAL_DIR = REPO_ROOT / 'evaluation'
EVAL_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = REPO_ROOT / 'runs' / 'mlp_best.pt'
SPLITS_DIR = REPO_ROOT / 'data' / 'splits'
LABELS_PATH = REPO_ROOT / 'data' / 'labels.json'
TRAINING_LOG = REPO_ROOT / 'runs' / 'training_log.csv'

print('device:', DEVICE)
print('checkpoint:', CKPT_PATH)
print('eval dir:', EVAL_DIR)

device: cpu
checkpoint: C:\Users\Harry T\Desktop\Hand_gesture_VL\runs\mlp_best.pt
eval dir: C:\Users\Harry T\Desktop\Hand_gesture_VL\evaluation


## 1. Load model, labels, test split, training log

Reuses `src.evaluate.load_model_checkpoint` so the same scaler that Stage 4 fit on train is applied here (no leakage).

In [2]:
from src.models.mlp import NUM_CLASSES

model, scaler, label_map, ckpt = ev.load_model_checkpoint(
    CKPT_PATH, labels_path=LABELS_PATH, device=DEVICE,
)
X_test, y_test = ev.load_test_split(SPLITS_DIR)
X_std = scaler.transform(X_test).astype(np.float32, copy=False)
X_t = torch.from_numpy(X_std)
y_t = torch.from_numpy(y_test.astype(np.int64, copy=False))
label_names = [label_map[i] for i in range(NUM_CLASSES)]

print(f'checkpoint epoch     : {ckpt["epoch"]}')
print(f'checkpoint val_loss  : {ckpt["val_loss"]:.4f}')
print(f'checkpoint val_acc   : {ckpt["val_acc"]:.4f}')
print(f'checkpoint merged_acc: {ckpt["merged_val_acc"]:.4f}')
print(f'test rows            : {X_test.shape[0]}, feature_dim {X_test.shape[1]}')

checkpoint epoch     : 76
checkpoint val_loss  : 0.0325
checkpoint val_acc   : 0.9910
checkpoint merged_acc: 0.9910
test rows            : 40895, feature_dim 279


## 2. Headline test metrics (from `runs/evaluation/test_metrics.json`)

Run `python -m src.evaluate` first if this file is missing. After the 26-class schema fix (see `tasks/peace_count2_collision_fix.md`), `merged_accuracy == accuracy` — the dual-labelling that previously bounded raw accuracy near 0.82 was removed at the schema level. The Stage 6 gate is on `merged_accuracy >= 0.90` and `macro_f1 >= 0.88`.

In [3]:
metrics_path = EVAL_DIR / 'test_metrics.json'
if not metrics_path.is_file():
    raise FileNotFoundError(
        f'{metrics_path} missing. Run: python -m src.evaluate'
    )
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
summary = {
    'raw accuracy': metrics['metrics']['accuracy'],
    'macro F1': metrics['metrics']['macro_f1'],
    'weighted F1': metrics['metrics']['weighted_f1'],
    'merged accuracy': metrics['metrics']['merged_accuracy'],
    'merged macro F1': metrics['metrics']['merged_macro_f1'],
}
print(pd.Series(summary).to_string(float_format=lambda v: f'{v:.4f}'))
print('\n' + metrics['acceptance']['gate_message'])

raw accuracy      0.9917
macro F1          0.9958
weighted F1       0.9917
merged accuracy   0.9917
merged macro F1   0.9958

GATE PASSED: merged_test_acc=0.9917 (>= 0.9: True); macro_f1=0.9958 (>= 0.88: True)


## 3. Plot 1 - Loss surface slice (level sets, MV 2.6)

Fix all parameters of the trained MLP at their checkpoint values **except** two scalar entries in `linears[0].weight`. Compute `L(w_a, w_b)` on a 50x50 grid centred at the trained values; render as a filled contour.

**Weight selection rule.** `select_active_first_layer_neuron` picks the layer-1 neuron with the highest mean ReLU activity on a probe batch (so the slice cuts through a part of the surface where small weight perturbations actually change the loss). We pair it with the input dimension whose mean `|x_j|` is largest. A literal `linears[0].weight[0, 0]` would land on a dead neuron in this checkpoint and give a pedagogically empty flat slice.

**Trajectory overlay.** `training_log.csv` logged Frobenius weight norms per layer, not per-weight trajectories. Reconstructing the projected trajectory would require re-running Stage 4 with extended logging - out of scope here. Only the trained point is overlaid.

In [4]:
probe = X_t[:512]
neuron_idx, input_idx_a = mv.select_active_first_layer_neuron(model, probe)
# Second input dim: deterministic, distinct from input_idx_a.
mean_abs = X_t[:512].abs().mean(dim=0).cpu().numpy()
order = np.argsort(-mean_abs)
input_idx_b = int([d for d in order if d != input_idx_a][0])

# Use a small, stratified-ish evaluation subset (first sample per class) so
# the 2500 grid evaluations finish in seconds while still covering every class.
first_per_class = []
for cls in range(NUM_CLASSES):
    idx = np.where(y_test == cls)[0]
    if idx.size:
        first_per_class.append(int(idx[0]))
eval_idx = np.array(first_per_class, dtype=int)
X_eval = X_t[eval_idx]
y_eval = y_t[eval_idx]

loss_payload = mv.compute_loss_surface_slice(
    model, X_eval, y_eval,
    weight_indices=((neuron_idx, input_idx_a), (neuron_idx, input_idx_b)),
    grid_size=50, radius=0.5, device=DEVICE,
)
print(f'neuron_idx={neuron_idx}, input dims ({input_idx_a}, {input_idx_b})')
print(f'trained loss (these {len(eval_idx)} samples): {loss_payload["trained_loss"]:.4f}')
print(f'grid loss range: [{loss_payload["loss_grid"].min():.4f}, {loss_payload["loss_grid"].max():.4f}]')

fig = mv.plot_loss_surface(
    loss_payload, output_path=EVAL_DIR / 'loss_surface_slice.png',
    title=f'Loss surface slice  -  W1[{neuron_idx},{input_idx_a}] vs W1[{neuron_idx},{input_idx_b}]',
)
plt.show()

neuron_idx=18, input dims (112, 52)
trained loss (these 26 samples): 0.0003
grid loss range: [0.0003, 0.0003]


C:\Users\Harry T\AppData\Local\Temp\ipykernel_38868\2083982744.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Reading the plot.** The black contours are level sets of `L` in this 2-D slice of the 105k-parameter loss surface. At any point `(w_a, w_b)`, the gradient `nabla L` is perpendicular to the contour it sits on, and gradient descent steps in the direction of `-nabla L` (steepest descent). The red star is where Stage 4 ended training. Concentric closed contours around the star are the local-minimum picture from MV 2.6.

## 3.5 Plot 5 - Loss landscape (filter-normalised random directions) with descent overlay

A broader 3D / contour view of the loss surface around `theta_best`,
plus a real **gradient descent trajectory** rolling down the surface
into the basin.

Where Plot 1 cuts a 2-coordinate axis-aligned slice through 2 of ~107k
parameters, Plot 5 picks **two random directions** in full parameter
space and rescales each `nn.Linear` row to match the trained-row L2 norm
(Li et al. 2018 *filter normalisation*). BatchNorm affine parameters are
held fixed.

**What this plot is.** Geometry of the cross-entropy loss in a random
2-D affine subspace through `theta_best`:

  $L_{\alpha,\beta}(\alpha, \beta) \;=\; L(\theta_{best} + \alpha\,d_1 + \beta\,d_2)$

The red trajectory is a *real* gradient descent on this 2-D restriction
of `L`, starting from a perturbed point and following $-\nabla L_{\alpha,\beta}$
until it reaches the basin around $\theta_{best}$.

**What this plot is NOT.**
- **Not the original Stage 4 trajectory.** Real training moved through
  full 107k-dimensional parameter space; the only checkpoint we still
  have is the endpoint $\theta_{best}$, so the actual descent path is
  unrecoverable from these artefacts. This trajectory shows what
  gradient descent looks like *on this 2-D slice*, started from a
  perturbed point on the same slice.
- **Not the full Hessian.** A random 2-D subspace can hide curvature
  that lives in the orthogonal complement. Sharpness / flatness
  conclusions here are relative, not absolute.

**Eval data.** A 4,000-sample stratified subsample of `data/splits/val.npz`
standardised by the Stage 4 scaler embedded in the checkpoint. Using val
(not test) keeps the test scorecard untouched.

In [5]:
from src import loss_landscape as ll
import time

VAL_NPZ = SPLITS_DIR / 'val.npz'

X_val_std, y_val = ll.select_eval_subset(
    VAL_NPZ,
    scaler_mean=ckpt['scaler_mean'],
    scaler_scale=ckpt['scaler_scale'],
    n_samples=ll.DEFAULT_EVAL_SUBSAMPLE,
    natural_only=False,
    seed=DEFAULT_SEED,
)
print(f'val subset: X {tuple(X_val_std.shape)}  y {tuple(y_val.shape)}')

d1, d2 = ll.generate_filter_normalized_directions(model, seed=DEFAULT_SEED)
ratios, dead = ll._per_linear_row_norm_ratios(model, d1)
print('filter-norm alive-row ratios per linear (should be ~1.0):', ratios)
print('dead rows per linear (rows whose trained norm is 0):', dead)

t0 = time.perf_counter()
landscape = ll.evaluate_loss_landscape(
    model, X_val_std, y_val, d1, d2,
    grid_size=ll.DEFAULT_GRID_SIZE,
    radius=ll.DEFAULT_RADIUS,
    device=DEVICE,
    eval_split='val',
    seed=DEFAULT_SEED,
)
dt = time.perf_counter() - t0

grid = landscape['loss_grid']
print(f'grid {grid.shape} computed in {dt:.1f}s')
print(f"anchor loss (theta_best on this subset): {landscape['anchor_loss']:.4f}")
print(f"grid loss range: [{grid.min():.4f}, {grid.max():.4f}]")

ll.save_landscape_npz(landscape, EVAL_DIR / 'loss_landscape_random.npz')
print('cached ->', EVAL_DIR / 'loss_landscape_random.npz')

# Gradient descent on the 2-D restriction L_{alpha,beta}, starting from a
# perturbed corner of the slice. This is a real descent on the real (but
# restricted) loss surface - NOT the original 107k-d Stage 4 trajectory.
descent = ll.compute_descent_path_2d(
    landscape,
    start=(-0.85, 0.85),
    learning_rate=0.18,
    n_steps=200,
    tolerance=1e-6,
)
print(f"descent: {descent['n_steps_taken']} steps from {descent['start']}"
      f" -> ({descent['path'][-1, 0]:+.3f}, {descent['path'][-1, 1]:+.3f}),"
      f" loss {descent['path'][0, 2]:.3f} -> {descent['path'][-1, 2]:.3f}"
      f" (converged: {descent['converged']})")

fig_c = ll.plot_landscape_contour(
    landscape,
    output_path=EVAL_DIR / 'loss_landscape_random_contour.png',
    descent_path=descent,
)
plt.show()

fig_s = ll.plot_landscape_surface(
    landscape,
    output_path=EVAL_DIR / 'loss_landscape_random_surface.png',
    descent_path=descent,
)
plt.show()

val subset: X (4000, 279)  y (4000,)
filter-norm alive-row ratios per linear (should be ~1.0): [1. 1. 1. 1.]
dead rows per linear (rows whose trained norm is 0): [71  0  0  0]


grid (51, 51) computed in 42.3s
anchor loss (theta_best on this subset): 0.0320
grid loss range: [0.0303, 2.3841]
cached -> C:\Users\Harry T\Desktop\Hand_gesture_VL\evaluation\loss_landscape_random.npz
descent: 124 steps from (-0.85, 0.85) -> (-0.245, +0.157), loss 0.666 -> 0.030 (converged: True)


C:\Users\Harry T\AppData\Local\Temp\ipykernel_38868\1385844149.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Harry T\AppData\Local\Temp\ipykernel_38868\1385844149.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Reading the plot.** The colour value at $(\alpha, \beta)$ is the
cross-entropy loss the model would have if its parameters were shifted by
$\alpha \cdot d_1 + \beta \cdot d_2$ in filter-normalised units. The
red star marks $\theta_{best}$ (the checkpoint) at $(0, 0)$.

The red marker-line is the gradient-descent path
$\theta_{t+1} = \theta_t - \eta \nabla L_{\alpha,\beta}(\theta_t)$
on the 2-D restriction of $L$, started from the black marker and ending
at the lime marker. Each step moves perpendicular to the contour it
sits on (steepest-descent direction in the slice) and shortens as the
gradient magnitude shrinks near the basin.

**Honest reading.** Two cautions for the audience:

1. **The trajectory is on the slice, not from training.** Stage 4
   gradient descent operated in 107k-d space and is not recoverable
   from a single checkpoint. This trajectory is what gradient descent
   on the 2-D restriction *would* look like from the chosen start;
   it shows the geometry of the basin, not the optimiser's history.
2. **The slice is one of infinitely many.** $d_1$ and $d_2$ are
   random with seed $\texttt{DEFAULT\_SEED}$; the basin width here is
   a typical 2-D cross-section, not an upper bound on Hessian
   curvature. Different seeds give different but qualitatively
   similar pictures (the network is well-trained).

**Mathematical safety.** The helper mutates parameters in place inside a
`torch.no_grad()` block and restores them in a `try/finally`. SHA-256 of
the model `state_dict` is asserted equal before and after the sweep, so
the trained weights cannot be silently corrupted by the visualisation.
The provenance fields written to `evaluation/loss_landscape_random.npz`
(seed, state-dict SHA-256, per-layer row-norm ratios, dead-row counts)
make the plot fully reproducible.

## 4. Plot 2 - Gradient norm vs. epoch (critical-point approach)

`grad_norm` is the L2 norm of all parameter gradients, captured on the last batch of each epoch *after* `loss.backward()` but *before* `optimizer.step()` - i.e. the gradient at the parameter point the next epoch starts from. As training approaches a critical point of `L`, this norm shrinks.

In [6]:
fig, log_df = mv.plot_gradient_norm(
    TRAINING_LOG,
    output_path=EVAL_DIR / 'grad_norm_vs_epoch.png',
    use_log_y=True,
)
print(f'epochs        : {len(log_df)}')
print(f'min grad_norm : {log_df["grad_norm"].min():.4f} at epoch {int(log_df["grad_norm"].idxmin())}')
print(f'best val_loss : {log_df["val_loss"].min():.4f} at epoch {int(log_df["val_loss"].idxmin())}')
plt.show()

epochs        : 77
min grad_norm : 0.0645 at epoch 72
best val_loss : 0.0325 at epoch 76


C:\Users\Harry T\AppData\Local\Temp\ipykernel_38868\3959676226.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Plot 3 - PCA of the 279-dim input space

Project the standardised test inputs onto the first two principal components. The model's decision boundaries are level sets of `f: R^279 -> R^26`; `nabla_x f` is perpendicular to those level sets. Visible class clustering here is what makes the engineered 279-feature representation a workable input for a small MLP.

In [7]:
pca_payload = mv.compute_input_pca(
    X_test, y_test,
    standardize_with=(np.asarray(ckpt['scaler_mean'], dtype=np.float32),
                       np.asarray(ckpt['scaler_scale'], dtype=np.float32)),
    subsample=5000, seed=DEFAULT_SEED,
)
evr = pca_payload['explained_variance_ratio']
print(f'explained variance: PC1={evr[0]:.3f}, PC2={evr[1]:.3f} (sum {evr.sum():.3f})')
print(f'subsample size    : {pca_payload["subsample_size"]}')

fig = mv.plot_input_pca(
    pca_payload, label_map,
    output_path=EVAL_DIR / 'pca_input_space.png',
)
plt.show()

explained variance: PC1=0.249, PC2=0.201 (sum 0.450)
subsample size    : 5000


C:\Users\Harry T\AppData\Local\Temp\ipykernel_38868\2631242955.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Plot 4 - Chain rule verification

For each held-out test sample, manually compute `dL/dW^(1)_{i,j}` using the explicit factorisation

$$
\frac{\partial L}{\partial W^{(1)}_{i,j}} \;=\; \underbrace{x_j}_{\text{input}} \cdot \underbrace{\frac{\gamma_i}{\sqrt{\sigma^2_i + \varepsilon}}}_{\text{BN scale}} \cdot \underbrace{\mathbf{1}[u^{(1)}_i > 0]}_{\text{ReLU gate}} \cdot \underbrace{\frac{\partial L}{\partial a^{(1)}_i}}_{\text{upstream grad}}
$$

in `eval()` mode (BatchNorm uses frozen running statistics, dropout is identity). Compare against `model.linears[0].weight.grad[i, j]` from `loss.backward()`. The plan's gate is `|manual - autograd| < 1e-5` on at least 10 distinct samples.

Sample indices are chosen via `pick_active_sample_indices` so the chosen neuron actually fires on most of them - the verification then shows all three non-trivial factors contributing, not a `0 = 0` triviality.

In [8]:
verify_indices = mv.pick_active_sample_indices(
    model, X_t, neuron_idx=neuron_idx, n_samples=12, seed=DEFAULT_SEED,
)
print(f'selected (neuron_idx, input_idx) = ({neuron_idx}, {input_idx_a})')
print(f'verification sample indices: {verify_indices}')

chain_df = mv.verify_chain_rule_batch(
    model, X_t, y_t, verify_indices,
    neuron_idx=neuron_idx, input_idx=input_idx_a,
    label_map=label_map, tol=mv.CHAIN_RULE_TOL_DEFAULT,
    required_passes=mv.CHAIN_RULE_REQUIRED_PASSES,
)
chain_path = EVAL_DIR / 'chain_rule_verification.csv'
chain_path.parent.mkdir(parents=True, exist_ok=True)
chain_df.to_csv(chain_path, index=False)

n_passed = int(chain_df['passed'].sum())
n_active = int(chain_df['gate_active'].sum())
max_err = float(chain_df['abs_error'].max())
print(f'chain rule passed: {n_passed}/{len(chain_df)}, max |error| = {max_err:.3e}')
print(f'gate-active samples: {n_active}/{len(chain_df)}')
chain_df[['sample_index', 'label_name', 'manual_grad', 'autograd_grad',
          'abs_error', 'gate_active', 'passed']]

selected (neuron_idx, input_idx) = (18, 112)
verification sample indices: [1471, 1577, 2264, 5760, 10199, 10272, 11163, 11460, 20056, 21433, 31495, 31781]
chain rule passed: 12/12, max |error| = 1.291e-11
gate-active samples: 12/12


,sample_index,label_name,manual_grad,autograd_grad,abs_error,gate_active,passed
0,1471,thumbs_up,-4.595792e-06,-4.595793e-06,4.627313e-13,True,True
1,1577,thumbs_up,-2.337291e-05,-2.337291e-05,1.889665e-13,True,True
2,2264,thumbs_down,2.672594e-05,2.672595e-05,3.403268e-12,True,True
3,5760,fist,1.351177e-06,1.351177e-06,3.534362e-14,True,True
4,10199,ok,-3.229681e-06,-3.229681e-06,5.378544e-14,True,True
5,10272,ok,-8.330467e-07,-8.330468e-07,1.078503e-13,True,True
6,11163,ok,-4.350055e-06,-4.350055e-06,5.979992e-14,True,True
7,11460,count_1,-5.463174e-05,-5.463174e-05,7.250506e-14,True,True
8,20056,call,-6.197995e-05,-6.197995e-05,4.055603e-13,True,True
9,21433,mute,3.657281e-04,3.657281e-04,1.290938e-11,True,True


## 7. Stage 6 acceptance summary

In [9]:
gate_rows = [
    {'gate': 'test accuracy (merged) >= 0.90',
     'value': metrics['metrics']['merged_accuracy'],
     'passed': metrics['acceptance']['test_accuracy_passed']},
    {'gate': 'macro F1 >= 0.88',
     'value': metrics['metrics']['macro_f1'],
     'passed': metrics['acceptance']['macro_f1_passed']},
    {'gate': 'chain rule passes >= 10 (tol 1e-5)',
     'value': int(chain_df['passed'].sum()),
     'passed': int(chain_df['passed'].sum()) >= mv.CHAIN_RULE_REQUIRED_PASSES},
]
summary_df = pd.DataFrame(gate_rows)
all_passed = bool(summary_df['passed'].all())
print(summary_df.to_string(index=False))
print('\nALL STAGE 6 GATES PASSED:' if all_passed else '\nGATE FAILURE:',
      all_passed)

                              gate     value  passed
    test accuracy (merged) >= 0.90  0.991735    True
                  macro F1 >= 0.88  0.995774    True
chain rule passes >= 10 (tol 1e-5) 12.000000    True

ALL STAGE 6 GATES PASSED: True


## 8. Interpretation for the class presentation

- **Loss surface (Plot 1).** The contour map is a 2-D slice of `L: R^P -> R` (P ~ 105k parameters). Level sets of `L` are exactly the contours you see; the gradient at any point on a contour is perpendicular to that contour and points toward steeper loss. Gradient descent walks down `-nabla L`. The red star marks where Stage 4 stopped.
- **Gradient norm (Plot 2).** `||nabla L||_2` decreasing across epochs is what "approaching a critical point" looks like in practice. At the best-val-loss epoch the norm has shrunk by an order of magnitude versus init.
- **PCA (Plot 3).** Even after collapsing 279 -> 2 dimensions, classes cluster - meaning the engineered feature map already separates many classes before the MLP applies its non-linear `f`. The decision boundaries the MLP learns are level sets of `f` in `R^279`, and `nabla_x f` is locally perpendicular to those boundaries.
- **Chain rule (Plot 4).** The recursive product `x_j * (gamma_i / sqrt(sigma^2_i + eps)) * gate_i * dL/da^(1)_i` matches PyTorch's `weight.grad` value to floating-point precision. Every backward pass in Stage 4 was running this same computation, in parallel, for every one of `P ~ 105k` parameters at every training step.